# Chapter 8 — Fading & Multipath — equations

Standalone, runnable subset of the master `../RF_Equations.ipynb`, scoped to this chapter.
Run top-to-bottom: **Setup**, then this chapter's sections. All functions are verified against the book's worked examples.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C = 2.99792458e8        # speed of light, m/s
EPS0 = 8.8541878128e-12 # vacuum permittivity, F/m

def wavelength(f_hz):
    return C / f_hz


## 7. Fresnel Zone Radius — Ch 8.2.2

$$r_n = \sqrt{\frac{n\,\lambda\,d_1 d_2}{d_1 + d_2}}$$

**Engine hook:** clearance test — if an obstacle intrudes past ~0.6·r₁ of the first zone,
the path is effectively obstructed and the diffraction model (next) kicks in.


In [ ]:
def fresnel_zone_radius(n, wavelength_m, d1_m, d2_m):
    d1_m = np.asarray(d1_m, float); d2_m = np.asarray(d2_m, float)
    return np.sqrt(n*wavelength_m*d1_m*d2_m/(d1_m + d2_m))

lam = wavelength(2.4e9)
D = 20.0
d1 = np.linspace(0.1, D-0.1, 200)
r1 = fresnel_zone_radius(1, lam, d1, D-d1)
plt.figure(figsize=(6,3.5))
plt.plot(d1, r1); plt.plot(d1, -r1, color=plt.gca().lines[-1].get_color())
plt.xlabel("position along path (m)"); plt.ylabel("1st Fresnel radius (m)")
plt.title(f"1st Fresnel zone, {D} m link @ 2.4 GHz"); plt.grid(True); plt.show()
print("max r1 =", r1.max(), "m at mid-path")


## 8. Knife-Edge Diffraction Loss — Ch 8.2.4

Fresnel–Kirchhoff parameter and the ITU single-edge loss approximation:

$$v = h\sqrt{\frac{2}{\lambda}\left(\frac1{d_1}+\frac1{d_2}\right)},\qquad
J(v)=6.9+20\log_{10}\!\left(\sqrt{(v-0.1)^2+1}+v-0.1\right)\ \text{dB},\ v>-0.78$$

`h` = obstruction height above the LOS line (negative if the edge is below LOS).

**Engine hook:** this is the diffraction effect — the extra loss that lets signal bend
into the geometric shadow behind corners/edges instead of leaving it black.


In [ ]:
def knife_edge_v(h_m, wavelength_m, d1_m, d2_m):
    return np.asarray(h_m, float)*np.sqrt(2.0/wavelength_m*(1.0/d1_m + 1.0/d2_m))

def knife_edge_loss_db(v):
    v = np.asarray(v, float)
    J = 6.9 + 20*np.log10(np.sqrt((v-0.1)**2 + 1) + v - 0.1)
    return np.where(v > -0.78, J, 0.0)

v = np.linspace(-3, 5, 400)
plt.figure(figsize=(6,4))
plt.plot(v, knife_edge_loss_db(v))
plt.axvline(0, ls="--", c="gray", label="grazing (v=0, ~6 dB)")
plt.xlabel("Fresnel–Kirchhoff v"); plt.ylabel("diffraction loss (dB)")
plt.title("Knife-edge diffraction (ITU approx)"); plt.legend(); plt.grid(True)
plt.gca().invert_yaxis(); plt.show()

# worked example: 0.5 m intrusion, mid-path of a 20 m link @ 2.4 GHz
lam = wavelength(2.4e9)
vv = knife_edge_v(0.5, lam, 10.0, 10.0)
print("v =", vv, "  loss =", float(knife_edge_loss_db(vv)), "dB")


## 15. Ground-Bounce Multipath, Roughness & Diffraction — Ch 8.2

Core physics for the **two-ray model** (Tier 1) and the **diffraction effect**. Extends the
pre-seeded §7 (Fresnel zones) and §8 (knife-edge) — both **verified below** against the book's
Examples 8.2 & 8.3 (seeded `fresnel_zone_radius` = eq 8.20, `knife_edge_v` = eq 8.19: exact match).

- **Two-ray ground bounce:** ρ ≈ −1 at small grazing angle ⇒ over flat earth received power falls
  as **1/d⁴** (not 1/d²), *independent of λ* (eq 8.13). Crosses over from FSL at `dx = 4π ht hr/λ`
  (eq 8.15); exact field oscillates as `2·sin(Δθ/2)` (nulls & the 6 dB constructive peak).
- **Rayleigh roughness** `HR = λ/(8 sinθ)` (eq 8.16): Δh < HR ⇒ smooth/specular (**reflection**);
  Δh ≫ HR ⇒ rough/diffuse (**scattering**). The reflection-vs-scattering trigger the engine needs.
- **Knife-edge** via Lee's piecewise approx (eqs 8.21) — matches the book; §8's ITU J(v) is an
  equivalent alternative. v = 0 ⇒ 6 dB (50 % blocked); v = −0.8 (60 % clearance) ⇒ 0 dB. Odd
  Fresnel-zone boundaries ⇒ destructive interference.


In [ ]:
def two_ray_reflection_point(ht, hr, d):
    d1 = d*ht/(hr + ht); return d1, d - d1                     # specular point (phi1 = phi2)

def two_ray_phase_diff(ht, hr, d, wavelength):                 # eq 8.8 (exact)
    return (2*np.pi/wavelength)*d*(np.sqrt(1+((hr+ht)/d)**2) - np.sqrt(1+((ht-hr)/d)**2))

def two_ray_crossover_m(ht, hr, wavelength):
    return 4*np.pi*ht*hr/wavelength                            # eq 8.15

def two_ray_pathloss_db(ht, hr, d, wavelength, gt=1.0, gr=1.0, exact=False):
    g_fsl = gt*gr*(wavelength/(4*np.pi*d))**2
    if exact:                                                  # eq 8.9: Lmp = Lfsl * 4 sin^2(dtheta/2)
        return -10*np.log10(g_fsl*4*np.sin(two_ray_phase_diff(ht,hr,d,wavelength)/2)**2)
    g_2ray = gt*gr*(ht*hr)**2/d**4                             # eq 8.13
    return -10*np.log10(min(g_2ray, g_fsl))                    # eq 8.14: use whichever gives greater loss

# Example 8.1: ht=hr=10 m, 2 GHz
lam2 = wavelength(2e9)
print(f"Ex 8.1: crossover dx = {two_ray_crossover_m(10,10,lam2):.0f} m")
print(f"  d=4 km  -> PL={two_ray_pathloss_db(10,10,4000,lam2):.1f} dB (< dx, so FSL; book 110.5)")
print(f"  d=40 km -> PL={two_ray_pathloss_db(10,10,40000,lam2):.1f} dB (> dx, so 1/d^4; book 144.1)")
print(f"  Fig 8.5 crossover (ht=100,hr=3,lam=0.3) = {two_ray_crossover_m(100,3,0.3)/1000:.1f} km (book 12.6)")


In [ ]:
def rayleigh_roughness_m(wavelength, grazing_rad):
    return wavelength/(8*np.sin(grazing_rad))                  # eq 8.16

def is_specular(delta_h_m, wavelength, grazing_rad):
    # True -> smooth/specular (reflection); False -> rough/diffuse (scattering).
    return delta_h_m < rayleigh_roughness_m(wavelength, grazing_rad)

g = np.radians(5)
hr_thr = rayleigh_roughness_m(wavelength(2.4e9), g)
print(f"Rayleigh HR @2.4 GHz, 5deg grazing = {hr_thr*100:.1f} cm  "
      f"(bumps below -> specular reflection, above -> diffuse scatter)")


In [ ]:
def knife_edge_loss_lee_db(v):
    # Lee's piecewise approximation to the diffraction integral (eqs 8.21). Loss in dB (>=0).
    v = np.asarray(v, float)
    with np.errstate(invalid="ignore", divide="ignore"):
        b1 = -20*np.log10(0.5 - 0.62*v)                        # -1 <= v <= 0
        b2 = -20*np.log10(0.5*np.exp(-0.95*v))                 #  0 <= v <= 1
        b3 = -20*np.log10(0.4 - np.sqrt(0.1184 - (0.38-0.1*v)**2))  # 1 <= v <= 2.4
        b4 = -20*np.log10(0.225/v)                             # v >= 2.4
    return np.select([v <= -1, v <= 0, v <= 1, v <= 2.4], [0.0, b1, b2, b3], default=b4)

# --- Verify the pre-seeded §7 & §8 functions against the book ---
# Example 8.2 (Fresnel zone): 1 km, 28 GHz, blockage 300 m from one end
r1 = fresnel_zone_radius(1, wavelength(28e9), 300, 700)
print(f"Ex 8.2: 1st Fresnel radius = {r1:.2f} m, 60% clearance = {0.6*r1:.2f} m (book ~0.9 m)")
# Example 8.3 (knife-edge): 150 MHz, edge 5 m below LOS, 200 m from one end
v83 = knife_edge_v(-5, wavelength(150e6), 200, 800)
print(f"Ex 8.3: v = {v83:.3f} (book -0.395),  Lee loss = {float(knife_edge_loss_lee_db(v83)):.1f} dB "
      f"(book 2.6),  ITU J(v) = {float(knife_edge_loss_db(v83)):.1f} dB")
print(f"        clearance check: v=-0.8 -> {float(knife_edge_loss_lee_db(-0.8)):.1f} dB, "
      f"v=0 -> {float(knife_edge_loss_lee_db(0.0)):.1f} dB")


## 16. Log-Normal Shadowing & Small-Scale Fading — Ch 8.3–8.5

The **statistical layer** on top of the deterministic median path loss. Two independent axes:
*large-scale* shadowing (log-normal, `X_σ`) and *small-scale* fading (Rayleigh/Ricean).

- **§8.3 Log-normal shadowing:** many diffraction/reflection losses multiply ⇒ add in dB ⇒
  (CLT) Gaussian-in-dB. Margin `L_S = z·σ_L` for a target coverage; `z = Φ⁻¹(coverage)`.
  Edge coverage `= Φ(M/σ_L)`. σ_L(Okumura fit) `= 0.65(log fc)² − 1.3 log fc + A` (A=5.2 urban /
  6.2 suburban). → `shadowing_margin_db()`, `edge_coverage_prob()`, `location_variability_okumura()`.
- **§8.4 Small-scale:** all-reflections ⇒ **Rayleigh**; a dominant/LOS path ⇒ **Ricean** (factor K).
  → `rayleigh_fade_prob()`, `ricean_fade_prob()`.
- **Delay spread** σ_τ ⇒ coherence BW `Bc ≈ 1/(5σ_τ)…1/(50σ_τ)` (eq 8.24); B<Bc = flat, else
  selective. **Doppler** `fm = Δv/λ` (eq 8.26) ⇒ coherence time `Tc ≈ 1/fm` (eq 8.25); T_sym<Tc = slow.
  → `coherence_bandwidth_hz()`, `doppler_shift_hz()`, `coherence_time_s()`.

*Engine:* layer `X_σ ~ N(0, σ_L)` onto any deterministic PL(x,y) for a coverage-*probability* map;
`log_distance_pl()` already carries σ. Delay-spread ties to the exponential impulse response of Ch 9.


In [ ]:
import math

def norm_cdf(x):                                   # Phi(x)
    return 0.5*(1 + np.vectorize(math.erf)(np.asarray(x, float)/math.sqrt(2)))

def norm_ppf(p):                                   # Phi^-1 (Acklam), |err| < 1.15e-9
    a=[-3.969683028665376e1,2.209460984245205e2,-2.759285104469687e2,1.383577518672690e2,-3.066479806614716e1,2.506628277459239e0]
    b=[-5.447609879822406e1,1.615858368580409e2,-1.556989798598866e2,6.680131188771972e1,-1.328068155288572e1]
    c=[-7.784894002430293e-3,-3.223964580411365e-1,-2.400758277161838e0,-2.549732539343734e0,4.374664141464968e0,2.938163982698783e0]
    d=[7.784695709041462e-3,3.224671290700398e-1,2.445134137142996e0,3.754408661907416e0]
    def one(p):
        if p < 0.02425:
            q=math.sqrt(-2*math.log(p));  return (((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5])/((((d[0]*q+d[1])*q+d[2])*q+d[3])*q+1)
        if p > 1-0.02425:
            q=math.sqrt(-2*math.log(1-p));return -(((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5])/((((d[0]*q+d[1])*q+d[2])*q+d[3])*q+1)
        q=p-0.5; r=q*q; return (((((a[0]*r+a[1])*r+a[2])*r+a[3])*r+a[4])*r+a[5])*q/(((((b[0]*r+b[1])*r+b[2])*r+b[3])*r+b[4])*r+1)
    return np.vectorize(one)(np.asarray(p, float))

def shadowing_margin_db(sigma_L, coverage):        # L_S = z * sigma_L
    return norm_ppf(coverage)*sigma_L

def edge_coverage_prob(margin_db, sigma_L):
    return norm_cdf(margin_db/sigma_L)

def location_variability_okumura(fc_mhz, area="urban"):
    A = 5.2 if area == "urban" else 6.2
    return 0.65*(np.log10(fc_mhz))**2 - 1.3*np.log10(fc_mhz) + A

# Example 8.5: 90% edge coverage
for s in (6, 8):
    print(f"Ex 8.5: sigma_L={s} dB -> z={norm_ppf(0.9):.2f}, shadowing margin L_S={shadowing_margin_db(s,0.9):.2f} dB "
          f"(book {7.7 if s==6 else 10.24})")


In [ ]:
def rayleigh_fade_prob(fade_db):
    # P(signal >= fade_db below the average), non-LOS multipath. Closed form (integrated Rayleigh pdf).
    return 1 - np.exp(-0.5*10**(-np.asarray(fade_db, float)/10))

def ricean_fade_prob(fade_db, K_dB, n=40000):
    # P(envelope power >= fade_db below the diffuse scale sigma^2) for a Ricean factor K (dB),
    # by numerical integration of the Ricean envelope pdf. Same reference as rayleigh_fade_prob,
    # so K -> -inf reduces to it. A dominant path (higher K) => FEWER deep fades than Rayleigh.
    # NOTE: the book's Ex 8.9 (0.018) used Mathcad with a different (unstated) fade reference and
    # does not reproduce cleanly; this value is internally consistent instead.
    Klin = 10**(K_dB/10.0); sig2 = 1.0
    A2 = 2*sig2*Klin; A = math.sqrt(A2)
    R = math.sqrt(sig2*10**(-fade_db/10.0))
    r = np.linspace(0, R, n)
    pdf = (r/sig2)*np.exp(-(r**2 + A2)/(2*sig2))*np.i0(A*r/sig2)
    return np.sum((pdf[:-1]+pdf[1:])/2*np.diff(r))   # trapezoid

print(f"Ex 8.8: Rayleigh P(>=12 dB fade) = {rayleigh_fade_prob(12):.3f} (book 0.031)")
print(f"Ex 8.9: Ricean P(>=12 dB fade), K=3 dB = {ricean_fade_prob(12, 3.0):.4f}  "
      f"(< Rayleigh's {rayleigh_fade_prob(12):.3f}: a dominant path steadies the signal; "
      f"book's 0.018 used a different reference)")


In [ ]:
def coherence_bandwidth_hz(rms_delay_spread_s, factor=5.0):
    return 1.0/(factor*rms_delay_spread_s)           # eq 8.24 (factor 5..50)

def delay_from_path_m(excess_path_m):
    return excess_path_m/C

def max_symbol_rate_from_delay(delay_s, fraction=0.1):
    return fraction/delay_s                          # delay <= fraction * T_sym

def doppler_shift_hz(velocity_ms, wavelength_m):
    return velocity_ms/wavelength_m                  # eq 8.26

def coherence_time_s(fm_hz, rms=False):
    return (0.423 if rms else 1.0)/fm_hz             # eq 8.27 / 8.25

# Example 8.6: 50 ksym/s -> B ~ 50 kHz = Bc = 1/(5 sigma_tau)
print(f"Ex 8.6: for Bc=50 kHz, rms delay spread = {1/(5*50e3)*1e6:.0f} us (book <= 4 us)")
# Example 8.7: direct 200 m, multireflection 235.2 m -> excess 35.2 m
tau = delay_from_path_m(35.2)
print(f"Ex 8.7: excess 35.2 m -> delay {tau*1e9:.0f} ns, max symbol rate {max_symbol_rate_from_delay(tau)/1e3:.0f} ksps (book 117 ns, 852)")
# Doppler: 30 m/s (108 km/h) at 2.4 GHz
print(f"Doppler @2.4 GHz, 30 m/s: fm={doppler_shift_hz(30, wavelength(2.4e9)):.0f} Hz, Tc={coherence_time_s(doppler_shift_hz(30, wavelength(2.4e9)))*1e3:.1f} ms")
